# WOURRI — Transcription Dioula via MMS-1B-ALL
**Vidéo Access Agriculture — toutes vidéos**

1. Installer les dépendances (+ noisereduce)
2. Uploader le fichier MP4
3. Découper en segments de 25s
4. Charger le modèle MMS-1B-ALL (dyi)
5. Transcrire avec débruitage + normalisation
6. Télécharger le fichier `.txt`

In [ ]:
# Cellule 1 — Installer les dépendances
!pip install -q transformers torch torchaudio accelerate noisereduce
!apt-get install -qq ffmpeg
print('✅ Dépendances installées')

In [ ]:
# Cellule 2 — Uploader le fichier audio MP4
from google.colab import files
import os

print('📂 Sélectionne le fichier MP4 (077 Lutter contre la cochenille...)')
uploaded = files.upload()

audio_path = list(uploaded.keys())[0]
print(f'✅ Fichier uploadé : {audio_path} ({os.path.getsize(audio_path) // 1024} KB)')

In [ ]:
# Cellule 3 — Découper l'audio en segments de 25s
import subprocess
import tempfile
import os

SEGMENT_S = 25
tmp_dir = '/content/segments'
os.makedirs(tmp_dir, exist_ok=True)

# Durée totale
result = subprocess.run(
    ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
     '-of', 'default=noprint_wrappers=1:nokey=1', audio_path],
    capture_output=True, text=True
)
duration = float(result.stdout.strip())
total = int(duration // SEGMENT_S) + 1
print(f'⏱️ Durée : {duration:.0f}s → {total} segments de {SEGMENT_S}s')

segments = []
for i in range(total):
    start = i * SEGMENT_S
    out_path = os.path.join(tmp_dir, f'seg_{i:04d}.wav')
    cmd = [
        'ffmpeg', '-y', '-loglevel', 'error',
        '-i', audio_path,
        '-ss', str(start), '-t', str(SEGMENT_S),
        '-ar', '16000', '-ac', '1',
        out_path
    ]
    subprocess.run(cmd, check=True)
    segments.append((i, start, out_path))

print(f'✅ {len(segments)} segments créés dans {tmp_dir}')

In [ ]:
# Cellule 4 — Charger le modèle MMS-1B-ALL (langue dyi = Dioula)
# ⚠️ Premier chargement : ~5-10 min (téléchargement ~4 GB)
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

LANGUAGE = 'dyi'   # Dioula (MMS-1B-ALL)
MODEL_ID  = 'facebook/mms-1b-all'

print(f'⏳ Chargement du modèle {MODEL_ID} pour {LANGUAGE}...')
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
model     = Wav2Vec2ForCTC.from_pretrained(MODEL_ID)

processor.tokenizer.set_target_lang(LANGUAGE)
model.load_adapter(LANGUAGE)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()
print(f'✅ Modèle chargé sur {device}')

In [ ]:
# Cellule 5 — Transcrire tous les segments
import torchaudio
import time

def transcribe_segment(seg_path, idx, total):
    import noisereduce as nr
    import numpy as np
    waveform, sr = torchaudio.load(seg_path)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
    waveform = waveform.squeeze().numpy()

    # Pré-traitement : débruitage + normalisation peak
    waveform = nr.reduce_noise(y=waveform, sr=16000, stationary=True, prop_decrease=0.75)
    peak = np.max(np.abs(waveform))
    if peak > 0:
        waveform = (waveform / peak * 0.95).astype('float32')

    inputs = processor(waveform, sampling_rate=16000, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_ids = torch.argmax(logits, dim=-1)[0]
    text = processor.decode(predicted_ids).strip()
    return text

transcriptions = []
print(f'🎙️ Transcription de {len(segments)} segments...\n')

for idx, start, seg_path in segments:
    mm = int(start // 60)
    ss = int(start % 60)
    print(f'  [{mm:02d}:{ss:02d}] Segment {idx+1}/{len(segments)}...', end=' ', flush=True)

    try:
        text = transcribe_segment(seg_path, idx, len(segments))
    except Exception as e:
        text = ''
        print(f'ERREUR: {e}', end=' ')

    transcriptions.append({'segment': idx, 'start_s': start,
                            'time': f'{mm:02d}:{ss:02d}', 'text': text})
    print(f'→ {text[:80] if text else "(vide)"}')

non_vides = sum(1 for t in transcriptions if t['text'])
print(f'\n✅ {non_vides}/{len(segments)} segments transcrits')

In [ ]:
# Cellule 6 — Sauvegarder et télécharger la transcription
from google.colab import files

base_name = os.path.splitext(os.path.basename(audio_path))[0]
output_file = f'/content/{base_name}_TRANSCRIPTION_DYI.txt'

with open(output_file, 'w', encoding='utf-8') as f:
    f.write(f'# Transcription Dioula — {base_name}\n')
    f.write(f'# Langue: {LANGUAGE} | Modèle: MMS-1B-ALL\n')
    f.write(f'# Segments: {len(segments)} × {SEGMENT_S}s\n\n')
    for t in transcriptions:
        if t['text']:
            f.write(f"[{t['time']}] {t['text']}\n")

print(f'📄 Fichier : {output_file}')
print('\n--- APERÇU (10 premiers segments) ---')
for t in transcriptions[:10]:
    if t['text']:
        print(f"[{t['time']}] {t['text']}")

print('\n⬇️ Téléchargement...')
files.download(output_file)